# Kokoro TextToSpeech Generator and TXT cleaner.

This is a tool that uses [Kokoro TTS](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX) and a [QWEN Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) models that are adapted to run in Google Colab, with no technical knowlage.

In [ ]:
# @title 🎙️ Kokoro TTS Generator 🎙️

# @markdown # Configure the TTS Generator!
# @markdown **Required Settings:**<br />
# @markdown **Input_Source:** How do you want to give your input? Direct or via .txt file? <br />
# @markdown **Save to Google Drive:** Check the box below to automatically save the final audio to your Google Drive. Either way it will be avilable for download from here.<br />
# @markdown **voice:** Select the voice you wish to use. All valid entities and samples can be found [HERE](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX#voicessamples) in the `Voices/Samples` section.<br />
# @markdown **<hr /> **The following options are For "text box" input_source only.** <br />**
# @markdown **text:** Put your text here. The box is physically small but you can put in any length! <br />
# @markdown **output_filename:** Select the Filename you want on your output, If you use the same name twice it will blindly REPLACE the old one. **Note:** When using .txt file, output file name is the name of the uploaded file.<br />
# @markdown Old generations can be found on the left of Colab in the "Files" section. <br />
# @markdown <hr />Note: If you need to clean up your text to improve the voice ouput quality a nice tool for that is just below!<hr />

input_source = "Upload .txt File" # @param ["Text Box", "Upload .txt File"]
# @markdown Save Location?
save_to_google_drive = False # @param {type:"boolean"}
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />
text = "Put your text here, This box is ignored if you are using a .txt file." # @param {type:"string"}
output_filename = "my_audio" # @param {type:"string"}

import os
import sys
import subprocess
import shutil
from IPython.display import Audio, display, clear_output

# --- BYPASS HUGGING FACE TOKEN POPUP ---
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"
# ---------------------------------------

# 1. Install Kokoro's native Python PyTorch library
try:
    import kokoro
    import soundfile as sf
except ImportError:
    print("Installing PyTorch Kokoro and dependencies (this takes a minute on the first run)...")
    subprocess.run(["pip", "install", "-q", "kokoro", "soundfile"], check=True)
    clear_output()
    import soundfile as sf

import torch
import numpy as np
from kokoro import KPipeline

# 2. Check for GPU Acceleration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Computing Device: {device.upper()}")
if device == 'cpu':
    print("⚠️ WARNING: GPU not detected. Generation will be slow. Go to Runtime > Change runtime type > select T4 GPU.")

# 3. Handle File Upload if selected
if input_source == "Upload .txt File":
    from google.colab import files
    print("\nPlease choose a .txt file to upload...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded. Stopping execution.")
        sys.exit()

    uploaded_filename = list(uploaded.keys())[0]
    with open(uploaded_filename, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"✅ Successfully read {len(text)} characters from {uploaded_filename}")

    # Overwrite the output_filename with the uploaded file's name (minus the extension)
    output_filename = os.path.splitext(uploaded_filename)[0]

# Ensure the filename ends with .wav
if not output_filename.endswith(".wav"):
    output_filename += ".wav"

# 4. Initialize the Model Pipeline
# The first letter of the voice dictates the language code ('a' for American, 'b' for British)
lang_code = voice[0]
print(f"\nLoading Kokoro-82M model onto {device.upper()}...")
pipeline = KPipeline(lang_code=lang_code, device=device)

# 5. Generate Audio
print(f"Generating speech for voice: {voice}...")

# The Python pipeline automatically handles sentence chunking internally!
generator = pipeline(text, voice=voice, speed=1.0)

audio_chunks = []
for i, (graphemes, phonemes, audio) in enumerate(generator):
    print(f"  -> Processed chunk {i+1}...")
    audio_chunks.append(audio)

# 6. Merge and Output
if audio_chunks:
    print("Merging chunks and saving file...")
    final_audio = np.concatenate(audio_chunks)

    # Save to Colab's temporary local environment
    sf.write(output_filename, final_audio, 24000)
    print(f"✅ Successfully created: {output_filename}")

    # Save to Google Drive if selected
    if save_to_google_drive:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')

        drive_path = f"/content/drive/MyDrive/{output_filename}"
        shutil.copy(output_filename, drive_path)
        print(f"💾 Successfully saved to Google Drive at: {drive_path}")

    display(Audio(output_filename, autoplay=True))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 🖨️ AI Text Cleaner 🖨️
# @markdown ### A tool that uses AI to clean up awfully formatted text. It may change the odd word, but will retain the same meaning. However it does this very infrequently.<br />
# @markdown Note: This is NOT recommended to process technical or mathmatical text were punctuation and format is critical.
# @markdown <br /><hr />
# @markdown **Required Settings:** <br /> <br />
# @markdown **input_source:** How do you want to give your input? Direct or via .txt file? <br />
# @markdown **Save to Google Drive:** Check the box below to automatically save the final text file to your Google Drive.<hr /><br />
# @markdown **These options are only used for "Text Box" input method.**<br /><br />
# @markdown **text:** If inputting via 'Text Box', paste that here. The box is physically small but you can put in any length!<br />
# @markdown **Output_filename:** Select the Filename you want on your output (Text Box mode only).<br />
# @markdown <br /> If you upload a .txt file, the output filename is ignored, and it uses the input file name and appends "_cleaned" instead.<br />

input_source = "Upload .txt File" # @param ["Text Box", "Upload .txt File"]
text = "Put your text here, This box is ignored if you are using a .txt file." # @param {type:"string"}
output_filename = "cleaned_document" # @param {type:"string"}
# @markdown Save Location?
save_to_google_drive = False # @param {type:"boolean"}

# --- BYPASS HUGGING FACE TOKEN POPUP ---
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"
# ---------------------------------------

# --- 1. Smart Initialization (Only runs on first click) ---
try:
    import transformers
except ImportError:
    print("📦 Installing required libraries... (First run only)")
    !pip install -q -U transformers accelerate torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import files
import os
import shutil
import textwrap

# Check if model is already in memory to save time on repeat runs
if 'model' not in globals() or 'tokenizer' not in globals():
    print("⏳ Loading Qwen2.5-3B-Instruct into GPU... (Takes 1-2 minutes)")
    model_name = "Qwen/Qwen2.5-3B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("✅ Model loaded successfully!")
else:
    print("⚡ Model already in memory. Skipping setup.")

# --- 2. AI Processing Functions ---
def process_text_with_ai(raw_text):
    system_prompt = (
        "You are an expert text editor. Your task is to clean up badly formatted text "
        "extracted from a PDF or OCR. "
        "Fix random line breaks, broken hyphenations, weird spacing, and remove inline "
        "headers, footers, or page numbers. Preserve the original meaning and structure. "
        "Output ONLY the cleaned text."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please clean the following text:\n\n{raw_text}"}
    ]

    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2000,
        temperature=0.1,
        do_sample=True,
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

def process_large_document(full_text, save_path, chunk_size=2500):
    """Breaks large text into manageable chunks and saves them continuously."""

    # Split text by words/spaces instead of newlines. This guarantees
    # it chops correctly even if the document is one giant unbroken string.
    words = full_text.replace('\n', ' \n ').split(' ')
    chunks = []
    current_chunk = ""

    for word in words:
        # If adding the next word keeps us under the limit, add it
        if len(current_chunk) + len(word) + 1 < chunk_size:
            current_chunk += word + " "
        else:
            # Otherwise, save the current chunk and start a new one
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = word + " "

    # Don't forget the last chunk!
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(f"🧩 Document safely split into {len(chunks)} manageable chunks.")

    # Initialize/Clear the output file before we start appending
    with open(save_path, "w", encoding="utf-8") as f:
        f.write("")

    for i, chunk in enumerate(chunks):
        print(f"⏳ Cleaning section {i+1} of {len(chunks)}...")
        cleaned_chunk = process_text_with_ai(chunk)

        # Open file in "append" mode ("a") to add the new chunk safely to disk
        with open(save_path, "a", encoding="utf-8") as f:
            f.write(cleaned_chunk + "\n\n")
            f.flush()            # Force Python to write the buffer
            os.fsync(f.fileno()) # Force the OS to write to disk immediately (crash protection)

# --- 3. Execution Logic ---
raw_input = ""
final_filename = f"{output_filename}.txt" # Default if using Text Box

if input_source == "Upload .txt File":
    print("📂 Awaiting file upload... Please select your .txt file below.")
    uploaded = files.upload()
    if not uploaded:
        print("❌ No file uploaded. Execution stopped.")
    else:
        # Get the first uploaded file
        original_filename = list(uploaded.keys())[0]
        raw_input = uploaded[original_filename].decode('utf-8')
        print(f"✅ Loaded {original_filename} successfully.")

        # Dynamically create the new filename based on the upload
        name_without_ext = os.path.splitext(original_filename)[0]
        final_filename = f"{name_without_ext}_cleaned.txt"
else:
    raw_input = text

# Clean the text if input exists
if raw_input.strip():
    print(f"✨ Starting AI cleaning process... Output will be saved to: {final_filename}")

    # Run the processing and save directly to final_filename dynamically
    process_large_document(raw_input, final_filename)

    print(f"🎉 Done! Cleaned text completely saved to: {final_filename}")

    # Save to Google Drive if selected
    if save_to_google_drive:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')

        drive_path = f"/content/drive/MyDrive/{final_filename}"
        shutil.copy(final_filename, drive_path)
        print(f"💾 Successfully saved to Google Drive at: {drive_path}")

    # Automatically trigger download of the cleaned file
    try:
        files.download(final_filename)
    except Exception as e:
        print(f"⚠️ Could not trigger automatic download. You can find '{final_filename}' in the folder icon on the left menu.")
else:
    print("⚠️ No text detected to clean. Please paste text in the box or upload a file.")